# Voxel-wise Permutation Testing demo

## Data simulation 

We simulate a null fMRI dataset with no true activation.  
For each subject ( i = 1, ... , n ), we generate a 2-D image Yi(x, y)  of size S*S (Currently S = 64) voxels.

### 1. Noise generation

Each voxel is drawn independently from a uniform distribution:

$$
Y_i(x, y) \sim \mathrm{Uniform}(-1, 1).
$$


### 2. Smoothing

To introduce spatial correlation, each image is convolved with a 
Gaussian smoothing kernel.

### 3. Final dataset

The simulated null dataset is:

$$
\{\, Y_i^{\text{smooth}}(x,y) \;:\; i=1,\dots,n \,\}
$$

with no true activation anywhere in the image.

# Models Used:

# One sample model (no labels)

Each voxel has observations:

$$
y = (y_1, y_2, \ldots, y_n)^\top.
$$

The design matrix is

$$
X =
\begin{bmatrix}
1 \\
1 \\
\vdots \\
1
\end{bmatrix}
\quad (n \times 1).
$$

The model is:

$$
y = X\beta + \varepsilon,
$$

where

- $\beta$ is the mean activation,  
- $\varepsilon \sim \mathcal{N}(0, \sigma^2 I)$.


The null hypothesis is

$$
H_0: \beta(v) = 0,
$$

that is, the mean signal at voxel \(v\) is zero.


The alternative hypothesis is

$$
H_1: \beta(v) \neq 0.
$$



### Contrast

$$
L = [1].
$$

### t statistic

$$
t(v) = \frac{L^\top \hat{\beta}(v)}{\sqrt{\hat{\sigma}^2(v)\, L^\top (X^\top X)^{-1} L}}.
$$


---

# Two sample model (with labels)

Let the subjects be in two equal groups.

The design matrix is

$$
X =
\begin{bmatrix}
1 & 0 \\
\vdots & \vdots \\
1 & 0 \\
0 & 1 \\
\vdots & \vdots \\
0 & 1
\end{bmatrix}
\quad (n \times 2).
$$

The model:

$$
y = X\beta + \varepsilon,
$$

with

$$
\beta =
\begin{bmatrix}
\mu_1 \\
\mu_2
\end{bmatrix},
\qquad
\varepsilon \sim \mathcal{N}(0, \sigma^2 I).
$$

### Contrast

$$
L = [\,1,\ -1\,].
$$

Tests:

$$
H_0:\ \mu_1 - \mu_2 = 0.
$$

### t statistic

$$
t(v) = 
\frac{L^\top \hat{\beta}(v)}
     {\sqrt{\hat{\sigma}^2(v)\, L^\top (X^\top X)^{-1} L}}.
$$


---

## Permutation Methods

### One sample (sign flipping)

Under the null:

$$
y_i^{new} = s_i\, y_i, \qquad s_i \in \{-1, +1\}.
$$

Each subject's sign is flipped independently with a 50:50 chance.


### Two sample (label shuffling)

Under the null, labels are exchangeable:

$$
X^{new} = \text{random permutation of rows of } X.
$$

This keeps the same number of subjects per group.


### Distribution of max T value

For each permutation

$$
m_p = \max_v \bigl|t_p(v)\bigr|.
$$


### Permutation threshold

$$
t_{\text{thr}} = \text{quantile}_{1-\alpha}(m_1,\dots,m_P).
$$


### Corrected significance map

$$
\text{sig}(v) = \mathbf{1}\{\,|t_{\text{real}}(v)| > t_{\text{thr}}\,\}.
$$

# FWER code demo

Import the helper function library

In [1]:
import helper_functions as hf

Initialise the parameters

In [ ]:
n_runs = 20 			# Number of simulation runs for FWER estimation
n_subj = 20  			# Number of subjects in each group
img_side_length = 64    # Side length of the square brain image in voxels
smoothing_sigma = 1.5 	# Standard deviation for Gaussian smoothing
alpha = 0.05      		# Significance level for FWER
n_perm = 500		    # Number of permutations for permutation test

Run the simulations and check the FWER error rate with no permutation correction applied

No labels (just 1 regressor in beta):

In [ ]:
fwer_false_pos_rate = hf.estimate_fwer(
    n_runs=n_runs,
    n_subj=n_subj,
    img_side=img_side_length,
    sigma=smoothing_sigma,
    alpha=alpha,
    labels=False
)

print(fwer_false_pos_rate)

With A/B labels (2 regressors in beta):

In [ ]:
fwer_false_pos_rate = hf.estimate_fwer(
    n_runs=n_runs,
    n_subj=n_subj,
    img_side=img_side_length,
    sigma=smoothing_sigma,
    alpha=alpha,
    labels=True
)

print(fwer_false_pos_rate)

In both cases, FWER is 1, which is very high, as expected.

# Permutation correction

With 2 labels:

In [ ]:
fwer_false_pos_rate = hf.estimate_fwer(
    n_runs=n_runs,
    n_subj=n_subj,
    img_side=img_side_length,
    sigma=smoothing_sigma,
    alpha=alpha,
    labels=True,
	n_perm=n_perm
)

print(fwer_false_pos_rate)

With 1 label:

In [ ]:
fwer_false_pos_rate = hf.estimate_fwer(
    n_runs=n_runs,
    n_subj=n_subj,
    img_side=img_side_length,
    sigma=smoothing_sigma,
    alpha=alpha,
    labels=False,
	n_perm=n_perm
)

print(fwer_false_pos_rate)

Results typically vary from 0.04 to 0.07, which is very reasonable. This is very close to the alpha = 0.05, and is a significant improvement over the non-controlled method.